# Create leases

In [5]:
from chi import lease, context
from datetime import datetime

def create_lease(type, site_name, node_type, amount, duration=None):
    if(duration is None):
        l = lease.Lease(
            name=f"{prefix}-{type}-{site_name}",
            start_date=start_date,
            end_date=end_date,
        )
    else:
        l = lease.Lease(
            name=f"{prefix}-{type}-{site_name}",
            start_date=start_date,
            duration=duration,
        )
    l.add_node_reservation(
        amount=amount,
        node_type=node_type
    )
    l.add_fip_reservation(
        amount=amount
    )
    l.submit(wait_for_active=False, idempotent=True)

def create_edge_lease(site_name, node_type, amount):
    l = lease.Lease(
            name=f"{prefix}-{site_name}",
            start_date=start_date,
            end_date=end_date,
        )
    l.add_node_reservation(
        amount=amount,
        node_type=node_type
    )
    l.add_fip_reservation(
        amount=amount
    )
    l.submit(wait_for_active=False, idempotent=True)

    
context.choose_project()

# Set the start and end times of the leases here
# If you want to start the leases immediately, set start_date = None below
start_year = 2026
start_month = 7
start_day = 20
start_hour = 10
start_minute = 0

end_year = 2026
end_month = 7
end_day = 27
end_hour = 9
end_minute = 55

start_date = datetime(start_year, start_month, start_day, start_hour, start_minute, 0)
#start_date = None
end_date = datetime(end_year, end_month, end_day, end_hour, end_minute, 0)

# Choose a prefix for all leases for automatic filtering
# Ensure that the prefix is not used by any other user of the same project
prefix = "ploner-llm-routing"


# Select node_types and amounts here. Leave type and site_names unchanged
# Minimal deployment: 
# site_name "uc": 1 router, 1 model 
# site_name "tacc": 1 router, 1 model 
# site_name "edge": 1 router, 1 model (amount=2 for edge because router and model have the same architecture)

skylake = "compute_skylake"
cascadelake = "compute_cascadelake_r"

context.use_site("CHI@UC")

create_lease(type="router", site_name="uc", node_type=skylake, amount=1)
create_lease(type="model", site_name="uc", node_type="gpu_rtx_6000", amount=1)
create_edge_lease(site_name="edge", node_type=skylake, amount=2)

context.use_site("CHI@TACC")

create_lease(type="router", site_name="tacc", node_type=skylake, amount=1)
create_lease(type="model", site_name="tacc", node_type="gpu_mi100", amount=1)

print("Created all leases")


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org
Now using project: CHI-261581


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org
Now using project: CHI-261581


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Created all leases


# Create instances

In [2]:
#hardware.show_nodes()
from chi import context
from chi import lease
from chi import server
from chi import container
import chi

context.choose_project()
sites = ["CHI@UC", "CHI@TACC"]
site_names = ["uc", "tacc"]

router_servers = []
model_servers = []
large_model_servers = []

prefix = "ploner-llm-routing"

start_edge_instances = False

print("Starting instances for UC and TACC...")

for site, site_name in zip(sites, site_names):
    print(f"======== Site: {site} ========")
    context.use_site(site)
    
    key_name = "chameleonKey"
    keypair = server.get_keypair(name="chameleonKey")
    
    leases = lease.list_leases()
    
    router_lease = None
    edge_lease = None
    model_lease = None
    large_model_lease = None
    
    for l in leases:
        if(not l.name.startswith(prefix) or l.status != "ACTIVE"):
            continue
        if("router" in l.name):
            router_lease = l
        if("edge" in l.name):
            edge_lease = l
        if("model" in l.name and "large" not in l.name):
            model_lease = l
        if("model-large" in l.name):
            large_model_lease = l
            
    router_server = None        

    large_model_server = None
    
    router_image = "CC-Ubuntu24.04"
    if(site_name == "edge"):
        router_image = "andreasploner/arm64-ubuntu-python"
        
    if(router_lease is not None):
        reservation = router_lease.node_reservations[0]
        
        server_name = f"{prefix}-router-{site_name}"
        
        router_server = server.Server(
            name=server_name,
            reservation_id=reservation["id"],
            image_name=router_image,
            key_name=key_name
        )
        router_server.submit(wait_for_active=False, idempotent=True)
        router_servers.append(router_server)
        print(f"Started {server_name}")
    else:
        print("No active router lease")
    
    model_image = "CC-Ubuntu24.04-CUDA"
    if(site_name == "tacc"):
        model_image = "CC-Ubuntu24.04-ROCm"
    if(site_name == "edge"):
        model_image = "andreasploner/arm64-ubuntu-python-ollama"
    if(model_lease is not None):
        reservations = model_lease.node_reservations
        print(reservations)
        
        reservation = model_lease.node_reservations[0]
        
        num_models = reservation["min"]
        print(num_models)
        
        for i in range(num_models):
            server_name = f"{prefix}-model-{site_name}-{i}"
            
            model_server = server.Server(
                name=server_name,
                reservation_id=reservation["id"],
                image_name=model_image ,
                key_name=key_name
            )
            model_server.submit(wait_for_active=False, idempotent=True)
            model_servers.append(model_server)
            print(f"Started {server_name}")
    else:
        print("No active model lease")
            
    if(large_model_lease is not None):
        reservation = large_model_lease.node_reservations[0]
        
        server_name = f"{prefix}-model-large-{site_name}"
        
        large_model_server = server.Server(
            name=server_name,
            reservation_id=reservation["id"],
            image_name=model_image,
            key_name=key_name
        )
        large_model_server.submit(wait_for_active=False, idempotent=True)
        large_model_servers.append(large_model_server)
        print(f"Started {server_name}")
    else:
        print("No active model-large lease")
        
    if(edge_lease is not None):
        reservations = edge_lease.node_reservations
        print(reservations)
        
        reservation = edge_lease.node_reservations[0]
        
        num_models = reservation["min"]
        print(num_models)
        
        for i in range(num_models):
            if(i == 0):
                server_name = f"{prefix}-router-edge"
            else:
                server_name = f"{prefix}-model-edge-{i}"
            
            edge_server = server.Server(
                name=server_name,
                reservation_id=reservation["id"],
                image_name=router_image,
                key_name=key_name
            )
            edge_server.submit(wait_for_active=False, idempotent=True)
            print(f"Started {server_name}")
        print(f"Started 1 edge router and {num_models - 1} edge models")
    else:
        print("No active edge lease")

if(not start_edge_instances):
    print("Skipping edge instances")
else:
    print("Starting instances for Edge...")
    context.use_site("CHI@Edge")
    context.use_project(project_id)
    site_name = "edge"

    edge_leases = lease.list_leases()
    for edge_lease in edge_leases:
        if(not edge_lease.name.startswith(prefix) or edge_lease.status != "ACTIVE"):
            continue

        reservation = edge_lease.device_reservations[0]

        num_models = reservation["min"]

        for i in range(num_models):
            if(i == 0):
                server_name = f"{prefix}-router-{site_name}"
                router_server = container.Container(
                    name=server_name,
                    reservation_id=reservation["id"],
                    image_ref="andreasploner/arm64-ubuntu-python",
                )
                router_server.submit(wait_for_active=False, idempotent=True)
                router_servers.append(router_server)
                print(f"Started {server_name}")
                continue

            server_name = f"{prefix}-model-{site_name}-{i}"
            model_server = container.Container(
                    name=server_name,
                    reservation_id=reservation["id"],
                    image_ref="andreasploner/arm64-ubuntu-python-ollama",
                )
            model_server.submit(wait_for_active=False, idempotent=True)
            model_servers.append(model_server)
            print(f"Started {server_name}")
        
print("Started all instances")

Starting instances for UC and TACC...
======== Site: CHI@UC ========
Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org
Now using project: CHI-261581


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Attribute,ploner-llm-routing-router-uc
Id,53bdfe1a-0199-457c-9fbe-bb8c9fb7cb59
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.140.83.24 (v4) Type: fixed MAC: 24:6e:96:7d:ff:bc
Network Name,sharednet1
Created At,2026-07-20T10:43:34Z
Keypair,chameleonKey
Reservation Id,None
Host Id,2dbc11e90237cd4cd3244aad5bae24b276f368727fc5e2c3cb127228


Started ploner-llm-routing-router-uc
[{'created_at': '2026-07-18 09:20:09', 'updated_at': '2026-07-20 10:00:07', 'id': 'b93408e6-389c-4ac2-a716-defe42365888', 'lease_id': '5d014868-0268-487a-b66f-d9a20a6da35c', 'resource_id': '414f0a24-536d-4977-8e13-fa7bf69e0795', 'resource_type': 'physical:host', 'status': 'active', 'missing_resources': False, 'resources_changed': False, 'hypervisor_properties': '', 'resource_properties': '["==", "$node_type", "gpu_rtx_6000"]', 'before_end': 'default', 'on_start': 'default', 'min': 1, 'max': 1}]
1


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


...

Attribute,ploner-llm-routing-router-tacc
Id,b99a1b23-dac0-4b97-8f36-19c864611aaa
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.0.107 (v4) Type: fixed MAC: 24:6e:96:62:a4:80
Network Name,sharednet1
Created At,2026-07-20T10:43:38Z
Keypair,chameleonKey
Reservation Id,None
Host Id,9d4bc0050a7e3dcaf3249dda065818f9e388eedea0fb8f26ef0a18f3


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Started ploner-llm-routing-router-tacc
[{'created_at': '2026-07-18 09:20:43', 'updated_at': '2026-07-20 10:00:07', 'id': '6f28a701-4660-4e9a-bfd0-4eeee290ef6c', 'lease_id': 'bc5fafb9-9e83-4be8-8275-b03583101ed2', 'resource_id': 'ae06f759-a66a-4206-9a44-b3136784a83a', 'resource_type': 'physical:host', 'status': 'active', 'missing_resources': False, 'resources_changed': False, 'hypervisor_properties': '', 'resource_properties': '["==", "$node_type", "gpu_mi100"]', 'before_end': 'default', 'on_start': 'default', 'min': 1, 'max': 1}]
1


Attribute,ploner-llm-routing-model-tacc-0
Id,None
Status,None
Image Name,CC-Ubuntu24.04-ROCm
Flavor Name,baremetal
Addresses,
Network Name,sharednet1
Created At,None
Keypair,chameleonKey
Reservation Id,6f28a701-4660-4e9a-bfd0-4eeee290ef6c
Host Id,None


Started ploner-llm-routing-model-tacc-0
No active model-large lease
No active edge lease
Skipping edge instances
Started all instances


# Check instance status

In [8]:
from chi import server, container
from prettytable import PrettyTable

server_names = []
server_status = []

context.use_site("CHI@UC")
servers = server.list_servers()
for s in servers:
    server_names.append(s.name)
    server_status.append(s.status)
    
context.use_site("CHI@TACC")
servers = server.list_servers()
for s in servers:
    server_names.append(s.name)
    server_status.append(s.status)
    
context.use_site("CHI@Edge")
containers = container.list_containers()
for s in containers:
    server_names.append(s.name)
    server_status.append(s.status)
t = PrettyTable(["Server", "Status"])
for name, status in zip(server_names, server_status):
    t.add_row([name, status])
print(t)


Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org
+---------------------------------+--------+
|              Server             | Status |
+---------------------------------+--------+
| ploner-llm-routing-model-edge-1 | BUILD  |
|  ploner-llm-routing-router-edge | ACTIVE |
|  ploner-llm-routing-model-uc-0  | ACTIVE |
|   ploner-llm-routing-router-uc  | ACTIVE |
| ploner-llm-routing-model-tacc-0 | ACTIVE |
|  ploner-llm-routing-router-tacc | ERROR  |
|            ekogl-rho            | ACTIVE |
+---------------------------------+--------+


# Create inventory.ini for Ansible

In [3]:
from chi import lease, context, server
import re

def replace_dash_with_underscore(s):
    return s.replace("-", "_")

project_id = "CHI-261581"
prefix = "ploner-llm-routing"
pattern = r"^ploner-llm-routing-(.+?)-([^-]+?)(?:-(\d+))?$"
mapping = {}

group_mapping = {
    "router-uc": "router",
    "router-tacc": "router",
    "router-edge": "router",
    "model-uc": "model_nvidia",
    "model-tacc": "model_amd",
    "model-edge": "model_cpu",
}

for cham_site in ["CHI@UC", "CHI@TACC"]:
    context.use_site(cham_site)
    context.use_project(project_id)

    leases = lease.list_leases()

    servers = server.list_servers()

    for l in leases:

        if(not l.name.startswith(prefix) or l.status != "ACTIVE"):
            continue
            
        print(f"----- {l.name} -----")

        floating_ips = l.get_reserved_floating_ips()
        print(floating_ips)

        reservation = l.node_reservations[0]
        dev_res = l.device_reservations

        #servers_of_lease = [s for s in servers if s.reservation_id == reservation["id"]]
        servers_of_lease = []
        for site in ["edge", "router-uc", "router-tacc", "model-uc", "model-tacc"]:
            if(site in l.name):
                for s in servers:
                    if( not (s.name.startswith(prefix) and site in s.name)):
                        continue
                    servers_of_lease.append(s)

        print(f"{len(servers_of_lease)} == {len(floating_ips)}")
        assert len(servers_of_lease) == len(floating_ips)
        for s, ip in zip(servers_of_lease, floating_ips):
            print(f"Assigning {ip} to {s.name}")
            s.associate_floating_ip(fip=ip)
            
            m = re.match(pattern, s.name)
            if(m):
                name = m.group(1)
                role = m.group(2)
                print(f"{name}, {role}")
                name = f"{name}-{role}"
                num = int(m.group(3)) if m.group(3) is not None else None
                print(f"{name} with id {num}")
                

                if(name in ["model-uc", "model-tacc"]):
                    num += 1
                ansible_name = name
                if(num is not None):
                    ansible_name = f"{name}-{num}"
                
                group = group_mapping[name]
                if(group not in mapping):
                    mapping[group] = []
                mapping[group].append({"ip": ip, "ansible_name": replace_dash_with_underscore(ansible_name), "ansible_group": group_mapping[name], "ansible_role": role})
            else:
                print(f"Did not match: {s.name}")


        print(l.name)
        print(servers_of_lease)
        print("="*50)


print()
print()
print()

for key, values in mapping.items():
    print(f"[{key}]")
    for value in values:
        print(f"{value['ansible_name']} ansible_host={value['ip']} ansible_user=cc role={value['ansible_role']}")
    print()

Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org
Now using project: CHI-261581


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The py

----- ploner-llm-routing-edge -----
['192.5.87.215', '192.5.86.219']
2 == 2
Assigning 192.5.87.215 to ploner-llm-routing-model-edge-1


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


model, edge
model-edge with id 1
Assigning 192.5.86.219 to ploner-llm-routing-router-edge


...

model, uc
model-uc with id 0
ploner-llm-routing-model-uc
[<Server 'ploner-llm-routing-model-uc-0'>]
----- ploner-llm-routing-router-uc -----
['192.5.87.202']
1 == 1
Assigning 192.5.87.202 to ploner-llm-routing-router-uc
router, uc
router-uc with id None
ploner-llm-routing-router-uc
[<Server 'ploner-llm-routing-router-uc'>]
Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org
Now using project: CHI-261581


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The py

----- ploner-llm-routing-model-tacc -----
['129.114.109.234']
1 == 1
Assigning 129.114.109.234 to ploner-llm-routing-model-tacc-0


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


model, tacc
model-tacc with id 0
ploner-llm-routing-model-tacc
[<Server 'ploner-llm-routing-model-tacc-0'>]
----- ploner-llm-routing-router-tacc -----
['129.114.109.220']
1 == 1
Assigning 129.114.109.220 to ploner-llm-routing-router-tacc
router, tacc
router-tacc with id None
ploner-llm-routing-router-tacc
[<Server 'ploner-llm-routing-router-tacc'>]
{'model_cpu': [{'ip': '192.5.87.215', 'ansible_name': 'model_edge_1', 'ansible_group': 'model_cpu', 'ansible_role': 'edge'}], 'router': [{'ip': '192.5.86.219', 'ansible_name': 'router_edge', 'ansible_group': 'router', 'ansible_role': 'edge'}, {'ip': '192.5.87.202', 'ansible_name': 'router_uc', 'ansible_group': 'router', 'ansible_role': 'uc'}, {'ip': '129.114.109.220', 'ansible_name': 'router_tacc', 'ansible_group': 'router', 'ansible_role': 'tacc'}], 'model_nvidia': [{'ip': '192.5.86.147', 'ansible_name': 'model_uc_1', 'ansible_group': 'model_nvidia', 'ansible_role': 'uc'}], 'model_amd': [{'ip': '129.114.109.234', 'ansible_name': 'model_tacc